# PatchCore on MVTec AD with layer-wise DINOv2 features

**CMPE297 final project – Finding the best DINO layer for fast anomaly detection**

This notebook runs the full pipeline for one **domain** (`"texture"` = carpet/tile/wood or
`"object"` = bottle/cable/transistor) and shows every helper from the `dinoad` package
step by step:

| Step | Helper(s) | Module |
|---|---|---|
| 1. Setup | – | – |
| 2. Config | – | – |
| 3. Data | `copy_to_local`, `verify_mvtec_root`, `MVTecAD`, `make_loader` | `dinoad.data` |
| 4. Backbone | `load_backbone` / `load_dinov2`, `build_extractor` (`ViTLayerExtractor`) | `dinoad.backbones` |
| 5. Features | `feature_cache_path`, `get_or_extract` (→ `extract_features`) | `dinoad.features` |
| 6. Memory bank | `PatchCoreConfig`, `PatchCore.fit`, `memory_bank_report` | `dinoad.patchcore`, `dinoad.efficiency` |
| 7. Inference | `PatchCore.predict` | `dinoad.patchcore` |
| 8. Accuracy | `image_auroc`, `pixel_auroc`, `pro_score`, `compute_metrics` | `dinoad.metrics` |
| 9. Efficiency | `benchmark_pipeline`, `benchmark_backbone_only` | `dinoad.efficiency` |
| 10. Visuals | `show_anomaly_maps`, `plot_score_histogram` | `dinoad.viz` |
| 11. Logging | `append_result` | `dinoad.results` |
| 12. Whole domain | `ExperimentConfig`, `run_domain` | `dinoad.experiment` |
| 13. Sweeps & summary | `run_category`, `load_results`, `domain_summary`, `plot_tradeoff` | `dinoad.experiment`, `dinoad.results`, `dinoad.viz` |

Run top to bottom. Change only the **Config** cell between runs.

## 1. Environment setup

* Select **Runtime → Change runtime type → GPU** (an L4 or T4 is plenty).
* Mount Google Drive, where the dataset, weights, feature cache and results live.
* Clone (or update) the team repo and install it in editable mode.

> The repo is private: store a GitHub token in Colab Secrets as `GH_TOKEN` and use the
> token URL shown in README section 2.1, or clone it once onto Drive and point `REPO_DIR` there.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/CraigBelshe/CMPE297_Project_dino-patchcore.git"  # private: use the token URL from README 2.1
REPO_DIR = "/content/CMPE297_Project_dino-patchcore" if IN_COLAB else os.path.abspath("..")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))   # makes the editable install importable without a restart

import torch, dinoad
print("dinoad", dinoad.__version__, "| torch", torch.__version__)
print("GPU:", dinoad.gpu_name())

## 2. Configuration

Everything you change between runs is here.

**Paths (on Drive)**
* `MVTEC_DRIVE` – folder holding either extracted category folders (`carpet/`, `bottle/`, ...)
  or MVTec's per-category archives (`carpet.tar.xz`, ...).
* `CACHE_DIR` – extracted features are saved here (fp16) and reused by later sweeps.
* `RESULTS_CSV` – one row per run is appended here; this is the team's shared results log.

**Backbone**
* `BACKBONE`: `"dinov2_vitb14"` (12 blocks, 768-d, patch 14), or the CNN baseline
  `"wide_resnet50_2"` / `"resnet50"` with `LAYERS = ["layer2", "layer3"]` and `IMAGE_SIZE = 224`.
* `DINO_SOURCE = "hub"` downloads public weights (cached in `HUB_DIR` on Drive so it only
  happens once). `"drive"` loads the state dict from `DINO_WEIGHTS`.
* `LAYERS` – **1-indexed** transformer blocks: `[12]` = last block, `[6]` = middle, `[6, 9]` = concatenation.

**PatchCore**
* `CORESET_RATIO` – fraction of training patches kept (1.0, 0.1, 0.05, 0.01).
* `REDUCER` / `REDUCER_DIM` – `"none"`, `"random"` (random projection) or `"pca"`.
  This, not the layer choice, is what shrinks the per-patch dimension.

**Resolution** – `IMAGE_SIZE` must be a multiple of 14 for DINOv2: 224 gives a 16×16 patch
grid, 448 gives 32×32 (better localisation, 4× more patches to store).

In [ ]:
# ---- paths -------------------------------------------------------------------
DRIVE_ROOT   = "/content/drive/MyDrive/CMPE297_final_project"
MVTEC_DRIVE  = f"{DRIVE_ROOT}/mvtec_anomaly_detection"
LOCAL_DATA   = "/content/mvtec"                 # fast VM disk; re-copied each session
CACHE_DIR    = f"{DRIVE_ROOT}/feature_cache"    # set to None to disable caching
RESULTS_CSV  = f"{DRIVE_ROOT}/results/results.csv"
OUTPUT_DIR   = f"{DRIVE_ROOT}/outputs"
USE_SYNTHETIC = False                           # True = fake data, only to test the pipeline

# ---- experiment ----------------------------------------------------------------
DOMAIN      = "texture"          # "texture" | "object"
CATEGORY    = None               # None = first category of the domain, for the step-by-step walkthrough
BACKBONE    = "dinov2_vitb14"
DINO_SOURCE = "hub"              # "hub" | "drive"
DINO_WEIGHTS  = f"{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth"   # used when DINO_SOURCE="drive"
DINO_REPO_DIR = None             # optional local clone of facebookresearch/dinov2
HUB_DIR       = f"{DRIVE_ROOT}/torch_hub"
DINO_PRETRAINED = True           # False only for offline pipeline tests
LAYERS      = [8]
IMAGE_SIZE  = 448
AGG_KERNEL  = 3                  # 3x3 neighbourhood aggregation (1 = off)
APPLY_NORM  = False              # apply final LayerNorm to intermediate tokens

CORESET_RATIO = 0.1
CORESET_METHOD = "greedy"        # "greedy" | "random"
REDUCER       = "none"           # "none" | "random" | "pca"
REDUCER_DIM   = None             # e.g. 256 or 128 when REDUCER != "none"
SEED          = 0

BATCH_SIZE  = 16
NUM_WORKERS = 2
BENCH_BATCH = 8
BENCH_ITERS = 20
RUN_SWEEP   = False              # section 13: set True to run the layer x coreset sweep

TAG = "walkthrough"              # free-text label saved with each results row

In [ ]:
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
categories = dinoad.get_categories(DOMAIN)          # e.g. ["carpet", "tile", "wood"]
CATEGORY = CATEGORY or categories[0]
print("device:", device, "| domain:", DOMAIN, categories, "| walkthrough category:", CATEGORY)

## 3. Data

**`copy_to_local(src_root, dst_root, categories)`** copies each category from Drive to the VM's
local disk (or extracts `<cat>.tar.xz`). Reading straight from Drive is several times slower.
Already-copied categories are skipped.

**`verify_mvtec_root(root, categories)`** counts train / test-good / test-anomalous images per
category and fails early if a folder is missing.

**`MVTecAD(root, category, split, image_size)`** is a `torch.utils.data.Dataset` returning dicts with
`image` (3×S×S, ImageNet-normalised), `mask` (1×S×S, 0/1), `label` (0 good / 1 anomalous),
`defect_type` and `path`. **`make_loader(ds, batch_size, num_workers)`** wraps it in an
unshuffled `DataLoader`, so outputs stay aligned with `ds.samples`.

In [ ]:
if USE_SYNTHETIC:
    from dinoad.synthetic import make_synthetic_mvtec
    DATA_ROOT = str(make_synthetic_mvtec(LOCAL_DATA + "_synthetic", categories))
else:
    DATA_ROOT = str(dinoad.copy_to_local(MVTEC_DRIVE, LOCAL_DATA, categories))

counts = dinoad.verify_mvtec_root(DATA_ROOT, categories)
for cat, c in counts.items():
    print(f"{cat:12s} {c}")

In [ ]:
from dinoad import MVTecAD, make_loader

train_ds = MVTecAD(DATA_ROOT, CATEGORY, split="train", image_size=IMAGE_SIZE)
test_ds  = MVTecAD(DATA_ROOT, CATEGORY, split="test",  image_size=IMAGE_SIZE)
train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader  = make_loader(test_ds,  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

item = test_ds[len(test_ds) - 1]
print(len(train_ds), "train /", len(test_ds), "test images")
print({k: (tuple(v.shape) if hasattr(v, "shape") else v) for k, v in item.items()})

## 4. Backbone and layer extractor

**`load_backbone(name, device, **kwargs)`** dispatches to:
* `load_dinov2(model_name, source="hub"|"drive", weights_path, repo_dir, hub_dir, pretrained)`
* `load_dinov3(model_name, repo_dir, weights_path)` (gated weights, patch size 16)
* `load_cnn(arch, weights_path)` for the ImageNet ResNet baseline

Models come back in `eval()` mode with gradients off.

**`build_extractor(model, backbone_name, layers, apply_norm, truncate=True)`** returns a
`ViTLayerExtractor` (or `CNNLayerExtractor`). It attaches **PyTorch forward hooks** to the
requested blocks for the duration of each call, strips the CLS/register tokens, and reshapes
tokens to a spatial map `(B, C, H/14, W/14)`. With `truncate=True` it **stops the forward pass
after the deepest requested block**, so layer 6 runs only 6 of 12 blocks. That is where the
throughput gain comes from. `extractor.compute_fraction` reports the share of blocks run.

In [ ]:
from dinoad import load_backbone, build_extractor

if BACKBONE.startswith("dinov2"):
    backbone_kwargs = dict(source=DINO_SOURCE, weights_path=DINO_WEIGHTS, repo_dir=DINO_REPO_DIR,
                           hub_dir=HUB_DIR, pretrained=DINO_PRETRAINED)
else:
    backbone_kwargs = {}
model = load_backbone(BACKBONE, device=device, **backbone_kwargs)
extractor = build_extractor(model, BACKBONE, LAYERS, apply_norm=APPLY_NORM, truncate=True)

with torch.no_grad():
    fmaps = extractor(item["image"][None].to(device))
for layer, f in fmaps.items():
    print(f"layer {layer}: {tuple(f.shape)}  (B, C, h, w)")
print(f"params: {dinoad.efficiency.count_params(model)/1e6:.1f}M | blocks executed: {extractor.compute_fraction:.0%}")

## 5. Patch features (with caching)

**`feature_cache_path(cache_dir, backbone, image_size, layers, category, split, agg_kernel, apply_norm)`**
builds a deterministic filename such as `feature_cache/dinov2_vitb14/448/carpet/train_L8_k3.pt`.

**`get_or_extract(cache_path, extractor, loader, device, agg_kernel, keep_masks)`** loads the cache if it
exists. Otherwise it calls **`extract_features`**, which runs the extractor, applies the 3×3
neighbourhood average (`local_aggregate`) and concatenates layers (`fuse_layers`), then saves fp16
tensors. It returns a **`FeatureSet`** with `.features (N, C, h, w)`, `.labels`, `.masks (N, S, S)`,
`.paths`, `.defect_types`.

Approximate cache size at 448 px: ~1.5 MB per image per 768-d layer.

In [ ]:
from dinoad import feature_cache_path, get_or_extract

def cache_for(split):
    if CACHE_DIR is None or USE_SYNTHETIC:
        return None
    return feature_cache_path(CACHE_DIR, BACKBONE, IMAGE_SIZE, LAYERS, CATEGORY, split, AGG_KERNEL, APPLY_NORM)

train_fs = get_or_extract(cache_for("train"), extractor, train_loader, device,
                          agg_kernel=AGG_KERNEL, keep_masks=False, desc="train")
test_fs  = get_or_extract(cache_for("test"), extractor, test_loader, device,
                          agg_kernel=AGG_KERNEL, desc="test")

print("train features:", tuple(train_fs.features.shape), train_fs.features.dtype)
print("test  features:", tuple(test_fs.features.shape), "| masks:", tuple(test_fs.masks.shape))
print(f"patches per image: {train_fs.grid[0]*train_fs.grid[1]} | feature dim per patch: {train_fs.dim}")

## 6. Build the memory bank (PatchCore fit)

**`PatchCoreConfig`** fields: `coreset_ratio`, `coreset_method` (`"greedy"` k-center or `"random"`),
`coreset_proj_dim` (projection used only to speed up coreset distance computations),
`reducer`/`reducer_dim` (stored dimension), `bank_dtype` (default fp16), `k_neighbors`, `blur_sigma`.

**`PatchCore(config, device).fit(train_features)`** flattens to a patch matrix, fits the reducer,
runs coreset selection, and stores the bank. `pc.fit_stats` records timings.

**`memory_bank_report(pc)`** reports storage: number of patches, dimension, dtype, MB (and the
fp32 equivalent), bytes per patch, and keep ratio.

In [ ]:
from dinoad import PatchCore, PatchCoreConfig, memory_bank_report

pc_cfg = PatchCoreConfig(coreset_ratio=CORESET_RATIO, coreset_method=CORESET_METHOD,
                         reducer=REDUCER, reducer_dim=REDUCER_DIM, seed=SEED)
pc = PatchCore(pc_cfg, device).fit(train_fs.features)

mem = memory_bank_report(pc)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in mem.items()})
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in pc.fit_stats.items()})

## 7. Inference

**`pc.predict(test_features, image_size)`** returns
* `scores` – `(N,)` image anomaly score = max patch distance to the memory bank;
* `maps` – `(N, S, S)` pixel anomaly maps (patch distances, bilinearly upsampled, Gaussian-blurred with σ=4).

Lower-level pieces, if you need them: `pc.score_patches(feats)` gives `(B, h, w)` nearest-neighbour
distances, and `pc.anomaly_map(patch_scores, S)` does the upsampling and blur.

In [ ]:
scores, maps = pc.predict(test_fs.features, IMAGE_SIZE)
print("scores:", scores.shape, "| maps:", maps.shape)

## 8. Accuracy metrics

* **`image_auroc(labels, scores)`** – detection: can the score separate good from defective images?
* **`pixel_auroc(masks, maps, max_pixels=None)`** – localisation over every test pixel
  (`max_pixels` subsamples for speed; leave `None` for reported numbers).
* **`pro_score(masks, maps, fpr_limit=0.3)`** – Per-Region Overlap. Each defect region counts
  equally regardless of size, integrated up to 30% FPR and normalised. Random ≈ 0.15, perfect = 1.
* **`compute_metrics(...)`** returns all three in one dict.

In [ ]:
from dinoad import image_auroc, pixel_auroc, pro_score, compute_metrics

labels, masks = test_fs.labels.numpy(), test_fs.masks.numpy()
print(f"image AUROC: {image_auroc(labels, scores):.4f}")
print(f"pixel AUROC: {pixel_auroc(masks, maps):.4f}")
print(f"PRO (FPR<=0.3): {pro_score(masks, maps):.4f}")

acc = compute_metrics(labels, scores, masks, maps)
acc

## 9. Efficiency metrics

**`benchmark_pipeline(extractor, pc, images, agg_kernel, n_warmup, n_iters)`** times test-time inference
on one batch, stage by stage (`backbone_ms`, `postproc_ms`, `knn_ms`, `map_ms`, all per image), plus
`total_ms`, `fps`, `peak_gpu_mb`, `compute_fraction` and `gpu`. It uses warmup runs, median timing
and `torch.cuda.synchronize()`.

**`benchmark_backbone_only(extractor, images)`** isolates backbone latency, which is what layer depth changes.

> Colab assigns GPUs dynamically. Only compare FPS between rows with the same `gpu` value.

In [ ]:
from dinoad import benchmark_pipeline, benchmark_backbone_only

bench_imgs = torch.stack([test_ds[i]["image"] for i in range(min(BENCH_BATCH, len(test_ds)))])
eff = benchmark_pipeline(extractor, pc, bench_imgs, agg_kernel=AGG_KERNEL, n_warmup=3, n_iters=BENCH_ITERS)
bb  = benchmark_backbone_only(extractor, bench_imgs.to(device), n_iters=BENCH_ITERS)
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in eff.items()})
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in bb.items()})

## 10. Visualisation

**`show_anomaly_maps(dataset, maps, scores, n=4, save_path=None)`** shows image | ground truth | heatmap
for the highest-scoring defective images (or pass `indices=`).
**`plot_score_histogram(labels, scores)`** shows how well image scores separate good from anomalous.

In [ ]:
from dinoad.viz import show_anomaly_maps, plot_score_histogram
import matplotlib.pyplot as plt

layer_tag = "+".join(map(str, LAYERS))
show_anomaly_maps(test_ds, maps, scores, n=3,
                  save_path=f"{OUTPUT_DIR}/{CATEGORY}_{BACKBONE}_L{layer_tag}_heatmaps.png")
plot_score_histogram(labels, scores, title=f"{CATEGORY} – {BACKBONE} L{layer_tag}")
plt.show()

## 11. Log the run

**`append_result(csv_path, row)`** appends one row (with a timestamp) to the shared CSV, widening the
header if new columns appear. Log every run, including failed or odd ones.

In [ ]:
from dataclasses import asdict
from dinoad import append_result, domain_of

row = {"category": CATEGORY, "domain": domain_of(CATEGORY), "backbone": BACKBONE,
       "layers": layer_tag, "image_size": IMAGE_SIZE, "agg_kernel": AGG_KERNEL, "apply_norm": APPLY_NORM,
       "feature_dim": train_fs.dim, **{f"pc_{k}": v for k, v in asdict(pc_cfg).items()},
       **acc, **mem, **pc.fit_stats, **eff, "tag": TAG}
if not USE_SYNTHETIC:
    append_result(RESULTS_CSV, row)
    print("appended to", RESULTS_CSV)

## 12. Run the whole domain

**`ExperimentConfig`** bundles everything above. **`run_domain(cfg, domain, model, device, results_csv)`**
calls **`run_category`** for every category in the domain (sections 3–9 in one call) and appends
each row to the CSV as soon as it finishes. If Colab disconnects, finished categories are kept,
and cached features make a rerun fast.

In [ ]:
from dinoad import ExperimentConfig, run_domain

exp_cfg = ExperimentConfig(
    data_root=DATA_ROOT, backbone=BACKBONE, layers=tuple(LAYERS), image_size=IMAGE_SIZE,
    agg_kernel=AGG_KERNEL, apply_norm=APPLY_NORM, patchcore=pc_cfg,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    cache_dir=None if USE_SYNTHETIC else CACHE_DIR,
    benchmark_batch=BENCH_BATCH, benchmark_iters=BENCH_ITERS, tag=TAG)

domain_rows = run_domain(exp_cfg, DOMAIN, model, device,
                         results_csv=None if USE_SYNTHETIC else RESULTS_CSV)

## 13. Sweeps and summary

Example sweep for experiments E1 and E3 (see `docs/EXPERIMENTS.md`): every layer × coreset ratio,
reusing one loaded backbone. Features are cached per layer, so only the first pass per layer runs
the backbone.

* **`run_category(cfg, category, model, device)`** returns one result row.
* **`load_results(csv)`** loads the shared CSV as a pandas DataFrame.
* **`domain_summary(df)`** gives mean metrics per configuration × domain.
* **`plot_tradeoff(df, x="bank_mb", y="pixel_auroc", hue="layers")`** plots accuracy vs storage (or `x="fps"`), one panel per domain.

In [ ]:
from dataclasses import replace
from dinoad import run_category, load_results, domain_summary
from dinoad.viz import plot_tradeoff

SWEEP_LAYERS  = [(3,), (6,), (8,), (9,), (12,)]
SWEEP_CORESET = [1.0, 0.1, 0.05, 0.01]

if RUN_SWEEP:
    for layers in SWEEP_LAYERS:
        for ratio in SWEEP_CORESET:
            cfg = replace(exp_cfg, layers=layers, tag="sweep_layer_coreset",
                          patchcore=replace(pc_cfg, coreset_ratio=ratio))
            for cat in categories:
                r = run_category(cfg, cat, model, device)
                append_result(RESULTS_CSV, r)
                print(layers, ratio, cat, f"pix-AUROC={r['pixel_auroc']:.4f} bank={r['bank_mb']:.1f}MB fps={r['fps']:.1f}")

if not USE_SYNTHETIC and Path(RESULTS_CSV).exists():
    df = load_results(RESULTS_CSV)
    display(domain_summary(df))
    plot_tradeoff(df, x="bank_mb", y="pixel_auroc", hue="layers",
                  save_path=f"{OUTPUT_DIR}/tradeoff_pixel_auroc_vs_bank.png")
    plt.show()
else:
    import pandas as pd
    display(pd.DataFrame(domain_rows)[["category", "domain", "layers", "image_auroc", "pixel_auroc",
                                       "pro", "bank_mb", "fps"]])